<br>

<img src="https://lindas.admin.ch/lindaslogo2.png" style="width:25%; float:right">

# LINDAS Tutorial

This page serves as a introductory tutorial to the LINDAS ecosystem. LINDAS is the Linked Data ecosystem of the Swiss Federal Administration.



# Introduction

The webpage you are currently viewing is a so called **interactive JupyterLite notebook**. In this notebook, you can change interactively the content of the single cells and execute these cells directly seeing the result of your changes immediately. The cells contain either markdown content (like this cell) or executable Python source code.

JupyterLite stems from JupyterLab with the advantage of being completely browser based without any backend infrastructure. This means that the execution of the cells could take some time during first execution. Subsequent executions will be much faster because of stored data in your browser cache.

If you are unfamiliar with the handling of Jupyter notebooks, here are two useful resources:

- [The JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [The Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

# Preliminaries

In this part some preliminaries for querying LINDAS with SPARQL are introduced. If you are only interested in the actual LINDAS tutorial, you can skip the whole section.

## Import the Necessary Modules

Querying a SPARQL endpoint is basically making a POST request to the corresponding endpoint URL. As JupyterLite at the moment has no support for Python's `requests` modul, the JavaScript fetch API is used (with some tricks). To make this happen, the following modules have to be importet: 

In [4]:
import json
import pandas as pd
from pyodide.ffi import to_js
from IPython.display import JSON, HTML
from js import Object, fetch
from io import StringIO

## Define Main Query Function

As the JavaScript fetch API is asynchronous, the corresponding Python function `query` has to be declared as `async`. This function allows to query the 3 Swiss governmental triple stores.

In [5]:
async def query(query_string, store = "L"):
    
    # three Swiss triplestores
    if store == "F":
        address = 'https://fedlex.data.admin.ch/sparqlendpoint'
    elif store == "G":
        address = 'https://geo.ld.admin.ch/query'
    else:
        address = 'https://ld.admin.ch/query'
    
    # try the Post request with help of JS fetch
    # the creation of the request header is a little bit complicated because it needs to be a 
    # JavaScript JSON that is made within a Python source code
    try:
        resp = await fetch(address,
          method="POST",
          body="query=" + query_string,
          credentials="same-origin",
          headers=Object.fromEntries(to_js({"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8", 
                                            "Accept": "text/csv" })),
        )
    except:
        raise RuntimeError("fetch failed")
    
    
    if resp.ok:
        res = await resp.text()
        # ld.admin.ch throws errors starting with '{"message":'
        if '{"message":' in res:
            error = json.loads(res)
            raise RuntimeError("SPARQL query malformed: " + error["message"])
        # geo.ld.admin.ch throws errors starting with 'Parse error:'
        elif 'Parse error:' in res:
            raise RuntimeError("SPARQL query malformed: " + res)
        else:
            # if everything works out, create a pandas dataframe from the csv result
            df = pd.read_csv(StringIO(res))
            return df
    else:
        # fedlex.data.admin.ch throws error with response status 400
        if resp.status == 400:
            raise RuntimeError("Response status 400: Possible malformed SPARQL query. No syntactic advice available.")
        else:
            raise RuntimeError("Response status " + resp.status)

If you are interested in the details of using the JavaScript fetch API within JupyterLite, please consult:

- https://pyodide.org/en/stable/usage/faq.html#how-can-i-use-fetch-with-optional-arguments-from-python
- https://github.com/jupyterlite/jupyterlite/discussions/412
- https://lwebapp.com/en/post/pyodide-fetch

## Define Display Function

Displays pandas dataframe resulting from the SPARQL query as HTML with clickable links.

In [6]:
def display_result(df):
    df = HTML(df.to_html(render_links=True, escape=False))
    display(df)

# Actual Tutorial

## Datasets

One of the basic elements in LINDAS are datasets that group similar data. The first task is to query LINDAS for all available datasets. Datasets in LINDAS have the [rdf:type](http://www.w3.org/1999/02/22-rdf-syntax-ns#type) [schema:Dataset](https://schema.org/Dataset).

In [9]:
df = await query("""

PREFIX schema: <http://schema.org/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

,dataset,name
0,https://culture.ld.admin.ch/.well-known/dataset/isil,ISIL-Kennzeichen
1,https://register.ld.admin.ch/.well-known/dataset/opendataswiss-meta,opendata.swiss Metadaten
2,https://energy.ld.admin.ch/elcom/electricityprice,Strompreis per Stromnetzbetreiber
3,https://energy.ld.admin.ch/elcom/electricityprice-canton,Median Strompreis per Kanton
4,https://energy.ld.admin.ch/elcom/electricityprice-swiss,Median Strompreis für die Schweiz
5,https://lod.opentransportdata.swiss/.well-known/dataset/didok,Dienststellen (DiDok)
6,https://lod.opentransportdata.swiss/.well-known/dataset/nova,NOVA
7,https://culture.ld.admin.ch/sfa/StateAccounts_Office/1/,Staatsrechnungen - Amt
8,https://culture.ld.admin.ch/sfa/StateAccounts_Domain/1/,Staatsrechnungen - Bereich
9,https://environment.ld.admin.ch/foen/ubd0037/1/,Lärmbelastung durch Verkehr


If you click on e.g. https://environment.ld.admin.ch/foen/ubd000504/3, you will get a useful webpage with additional infos on this dataset. For example, the [purl:creator](http://purl.org/dc/terms/creator) https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu is stated. We can use this to refine our query for showing only datasets from the BAFU:

In [10]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>

SELECT * WHERE {
    ?dataset a schema:Dataset;
        schema:name ?name;
        purl:creator <https://register.ld.admin.ch/opendataswiss/org/bundesamt-fur-umwelt-bafu>.
        
    FILTER(lang(?name) = 'de')
}

""")

display_result(df)

,dataset,name
0,https://environment.ld.admin.ch/foen/ubd0037/1/,Lärmbelastung durch Verkehr
1,https://environment.ld.admin.ch/foen/ubd0104/1/,Qualität der Badegewässer
2,https://environment.ld.admin.ch/foen/ubd0066/1/,Schwermetallbelastung des Bodens
3,https://environment.ld.admin.ch/foen/ubd0066/2/,Schwermetallbelastung des Bodens
4,https://environment.ld.admin.ch/foen/ubd0037/2/,Lärmbelastung durch Verkehr
5,https://environment.ld.admin.ch/foen/ubd0066/3/,Schwermetallbelastung des Bodens
6,https://environment.ld.admin.ch/foen/ubd0104/2/,Qualität der Badegewässer
7,https://environment.ld.admin.ch/foen/ubd0037/3/,Lärmbelastung durch Verkehr
8,https://environment.ld.admin.ch/foen/ubd0066/4/,Schwermetallbelastung des Bodens
9,https://environment.ld.admin.ch/foen/ubd0104/3/,Qualität der Badegewässer


In [13]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {
    <https://environment.ld.admin.ch/foen/ubd0104/6> cube:observationSet/cube:observation ?observation.
    ?observation <https://environment.ld.admin.ch/foen/ubd0104/dateofprobing> ?dateOfProbing;
        <https://environment.ld.admin.ch/foen/ubd0104/monitoringprogramm> ?monitoringProgramm;
        <https://environment.ld.admin.ch/foen/ubd0104/parametertype> ?parameterType;
        <https://environment.ld.admin.ch/foen/ubd0104/station> ?station;
        <https://environment.ld.admin.ch/foen/ubd0104/value> ?value.
}

LIMIT 100

""")

display_result(df)

,observation,dateOfProbing,monitoringProgramm,parameterType,station,value
0,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/E.coli/CH10001,2019-05-23,BAQUA_FR,E.coli,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,210
1,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/E.coli/CH10001,2020-05-26,BAQUA_FR,E.coli,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,80
2,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-05-23/Enterokokken/CH10001,2019-05-23,BAQUA_FR,Enterokokken,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,62
3,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-08-13/Enterokokken/CH10001,2019-08-13,BAQUA_FR,Enterokokken,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,31
4,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/E.coli/CH10001,2020-08-11,BAQUA_FR,E.coli,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,21
5,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2019-07-24/E.coli/CH10001,2019-07-24,BAQUA_FR,E.coli,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,17
6,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2018-09-03/E.coli/CH10001,2018-09-03,BAQUA_FR,E.coli,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,15
7,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-06-23/Enterokokken/CH10001,2020-06-23,BAQUA_FR,Enterokokken,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,14
8,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-05-26/Enterokokken/CH10001,2020-05-26,BAQUA_FR,Enterokokken,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,14
9,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2020-08-11/Enterokokken/CH10001,2020-08-11,BAQUA_FR,Enterokokken,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,12


In [42]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT DISTINCT ?station ?canton ?name ?latitude ?longitude WHERE {
    <https://environment.ld.admin.ch/foen/ubd0104/6> cube:observationSet/cube:observation ?observation.
    ?observation <https://environment.ld.admin.ch/foen/ubd0104/station> ?station.
    ?station <https://environment.ld.admin.ch/foen/ubd0104/kanton> ?canton;
        schema:name ?name;
        schema:latitude ?latitude;
        schema:longitude ?longitude.
}

""")

display_result(df)

,station,canton,name,latitude,longitude
0,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10001,FR,Plage de Gumefens,46.6753,7.0879
1,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10004,FR,Gemeinde Strandbad,46.9279,7.1109
2,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10007,FR,Plage de Portalban,46.9227,6.9534
3,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH10009,FR,Nouvelle plage,46.8571,6.8483
4,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1013,ZH,Strandbad Maur,47.3456,8.6747
5,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1016,ZH,Strandbad Uster,47.3414,8.6911
6,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1029,ZH,Strandbad Baumen,47.3583,8.7855
7,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1050,ZH,Strandbad Küsnacht,47.3093,8.5826
8,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1058,ZH,Seebad Lattenberg,47.2390,8.7171
9,https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH1061,ZH,Strandbad Bürger,47.2891,8.5749


In [41]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX purl: <http://purl.org/dc/terms/>
PREFIX cube: <https://cube.link/>

SELECT * WHERE {
    <https://environment.ld.admin.ch/foen/ubd0104/6> cube:observationSet/cube:observation ?observation.
    ?observation <https://environment.ld.admin.ch/foen/ubd0104/dateofprobing> ?dateOfProbing;
        <https://environment.ld.admin.ch/foen/ubd0104/monitoringprogramm> ?monitoringProgramm;
        <https://environment.ld.admin.ch/foen/ubd0104/parametertype> ?parameterType;
        <https://environment.ld.admin.ch/foen/ubd0104/station> <https://environment.ld.admin.ch/foen/ubd0104/6/Station/CH21055>;
        <https://environment.ld.admin.ch/foen/ubd0104/value> ?value.
}

LIMIT 100

""")

display_result(df)

,observation,dateOfProbing,monitoringProgramm,parameterType,value
0,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2011-07-11/Enterokokken/CH21055,2011-07-11,BAQUA_TI,Enterokokken,630
1,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2016-07-14/E.coli/CH21055,2016-07-14,BAQUA_TI,E.coli,100
2,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2013-05-24/Enterokokken/CH21055,2013-05-24,BAQUA_TI,Enterokokken,90
3,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2016-05-17/E.coli/CH21055,2016-05-17,BAQUA_TI,E.coli,50
4,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2007-06-19/E.coli/CH21055,2007-06-19,BAQUA_TI,E.coli,40
5,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2013-05-24/E.coli/CH21055,2013-05-24,BAQUA_TI,E.coli,40
6,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2016-07-14/Enterokokken/CH21055,2016-07-14,BAQUA_TI,Enterokokken,40
7,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2017-08-09/E.coli/CH21055,2017-08-09,BAQUA_TI,E.coli,40
8,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2011-07-21/E.coli/CH21055,2011-07-21,BAQUA_TI,E.coli,30
9,https://environment.ld.admin.ch/foen/ubd0104/6/observation/2017-08-09/Enterokokken/CH21055,2017-08-09,BAQUA_TI,Enterokokken,30


In [24]:
%pip install -q folium
%pip install -q geopandas

In [25]:
import folium
import geopandas

m = folium.Map(location=[47.1329, 8.6106], zoom_start=17.0)
m

In [31]:
geometry = geopandas.points_from_xy(df.longitude, df.latitude)
geo_df = geopandas.GeoDataFrame(df[["canton", "name"]], geometry=geometry)
geo_df.head()

,canton,name,geometry
0,FR,Plage de Gumefens,POINT (7.08790 46.67530)
1,FR,Gemeinde Strandbad,POINT (7.11090 46.92790)
2,FR,Plage de Portalban,POINT (6.95340 46.92270)
3,FR,Nouvelle plage,POINT (6.84830 46.85710)
4,ZH,Strandbad Maur,POINT (8.67470 47.34560)


In [52]:
m = folium.Map(location=[df['latitude'].mean(),df['longitude'].mean()], tiles="OpenStreetMap", zoom_start=7)

# add marker one by one on the map
for i in range(0,len(df)):
   folium.Marker(
      location=[df.iloc[i]['latitude'], df.iloc[i]['longitude']],
      popup=folium.Popup(folium.Html('<a href="' + df.iloc[i]['station'] + '" target="_blank">' + df.iloc[i]['name'] + '</a>', script=True), max_width=500),
    icon=folium.Icon(icon='person-swimming', prefix='fa')
   ).add_to(m)

# Show the map again
m

In [56]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT * WHERE {
	?DefinedTermSet a schema:DefinedTermSet;
        schema:name ?Name.
  	FILTER(regex(str(?DefinedTermSet), "admin.ch" ) )
    FILTER(lang(?Name) = "de")
}

""")

display_result(df)

,DefinedTermSet,Name
0,https://culture.ld.admin.ch/isil,ISIL-Kennzeichen
1,https://register.ld.admin.ch/opendataswiss/category,Kategorien von opendata.swiss
2,https://ld.admin.ch/ech/97/legalforms,Rechtsformen
3,https://ld.admin.ch/dimension/zefix,Zefix - Zentraler Firmenindex
4,https://ld.admin.ch/dimension/department,Eidgenössische Departemente und die Bundeskanzlei
5,https://ld.admin.ch/dimension/office,Bundesämter
6,https://ld.admin.ch/dimension/canton,Kantone
7,https://ld.admin.ch/dimension/municipality,Gemeinden
8,https://ld.admin.ch/cube/dimension/el01,Schwermetall
9,https://register.ld.admin.ch/opendataswiss/org,Die Organisationen von opendata.swiss


In [62]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX gn: <http://www.geonames.org/ontology#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?Municipality ?Name ?Population WHERE {
    ?Municipality gn:featureCode gn:A.ADM3 .
    ?Municipality schema:name ?Name .
    ?Municipality gn:population ?Population .
    ?Municipality <http://purl.org/dc/terms/issued> ?Date .
    FILTER (?Date = "2022-01-01"^^xsd:date)
}
ORDER BY DESC(?Population)
LIMIT 5

""", "G")

display_result(df)

,Municipality,Name,Population
0,https://geo.ld.admin.ch/boundaries/municipality/261:2022,Zürich,421878
1,https://geo.ld.admin.ch/boundaries/municipality/6621:2022,Genève,203856
2,https://geo.ld.admin.ch/boundaries/municipality/2701:2022,Basel,173863
3,https://geo.ld.admin.ch/boundaries/municipality/5586:2022,Lausanne,140202
4,https://geo.ld.admin.ch/boundaries/municipality/351:2022,Bern,134794


In [61]:
df = await query("""

PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX jolux: <http://data.legilux.public.lu/resource/ontology/jolux#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?eventLabel ?decisionDate ?sourceId  WHERE {
  ?actAbstract jolux:classifiedByTaxonomyEntry ?concept .
  ?concept skos:notation ?notation .
  filter(datatype(?notation) = <https://fedlex.data.admin.ch/vocabulary/notation-type/id-systematique>)
  filter(str(?notation) = '170.512')

  ?actAbstract  jolux:basicAct ?basicAct .
  ?draft jolux:hasResultingLegalResource ?basicAct .
  ?draft rdf:type jolux:InitialDraft .

  ?draft jolux:draftHasLegislativeTask ?event . 
  ?event jolux:legislativeTaskType ?eventType .
  ?eventType skos:prefLabel ?eventLabel . filter(lang(?eventLabel) = 'fr')
  ?event jolux:decisionDate ?decisionDate .
  optional {
    ?event jolux:legislativeTaskHasResultingLegalResource ?source .
    ?source jolux:isRealizedBy ?expression .
    ?expression jolux:historicalLegalId ?sourceId .
    ?expression jolux:language <http://publications.europa.eu/resource/authority/language/FRA> .
  }
} 
order by desc(?decisionDate)

""", "F")

display_result(df)

,eventLabel,decisionDate,sourceId
0,Expiration du délai référendaire,2004-10-07,NaN
1,Arrêté du Parlement,2004-06-18,FF 2004 2919
2,Message du Conseil fédéral,2003-10-22,FF 2003 7047
3,Arrêté du Parlement,1986-03-21,FF 1986 I 858
4,Message du Conseil fédéral,1983-06-29,FF 1983 III 441
